In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json, os

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

PLOT_DIR = r'c:\ML\EXP8\plots'
os.makedirs(PLOT_DIR, exist_ok=True)

df = pd.read_csv(r'c:\ML\EXP8\Dataset.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Missing values:\n{df.isnull().sum()}")

# Regression target
y_reg = df['Final_Score_Regression'].values
# Classification target
le = LabelEncoder()
y_cls = le.fit_transform(df['Performance_Level_Classification'])
print(f"Classification classes: {le.classes_}")
print(f"Class distribution: {np.bincount(y_cls)}")

# Features (exclude both targets)
X_raw = df.drop(columns=['Final_Score_Regression', 'Performance_Level_Classification']).values
print(f"Features shape: {X_raw.shape}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# PCA to 95% variance
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_.sum() * 100
n_components = X_pca.shape[1]
print(f"PCA: {n_components} components explain {explained:.2f}% of variance")

# Scree Plot
fig, ax = plt.subplots(figsize=(8, 5))
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
ax.plot(range(1, n_components+1), cumvar, 'bo-', linewidth=2)
ax.axhline(95, color='r', linestyle='--', label='95% threshold')
ax.set_xlabel('Number of Components'); ax.set_ylabel('Cumulative Explained Variance (%)')
ax.set_title('PCA Scree Plot'); ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'pca_scree.png'))
plt.savefig(os.path.join(PLOT_DIR, 'pca_scree.eps'))
plt.close()
print("Saved scree plot.")

cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)
cv_cls = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def neg_mse_to_mse(scores): return -scores

In [ ]:
# ── REGRESSION ────────────────────────────────────────────────────────────────
# Linear Regression
lr = LinearRegression()
lr_mse_nopca = neg_mse_to_mse(cross_val_score(lr, X_scaled, y_reg, cv=cv_reg, scoring='neg_mean_squared_error'))
lr_r2_nopca  = cross_val_score(lr, X_scaled, y_reg, cv=cv_reg, scoring='r2')
lr_mse_pca   = neg_mse_to_mse(cross_val_score(lr, X_pca, y_reg, cv=cv_reg, scoring='neg_mean_squared_error'))
lr_r2_pca    = cross_val_score(lr, X_pca, y_reg, cv=cv_reg, scoring='r2')
print(f"LR  NoPCA: MSE={lr_mse_nopca.mean():.4f} R2={lr_r2_nopca.mean():.4f}")
print(f"LR   PCA:  MSE={lr_mse_pca.mean():.4f} R2={lr_r2_pca.mean():.4f}")

# Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1)
rf_mse_nopca = neg_mse_to_mse(cross_val_score(rf, X_scaled, y_reg, cv=cv_reg, scoring='neg_mean_squared_error'))
rf_r2_nopca  = cross_val_score(rf, X_scaled, y_reg, cv=cv_reg, scoring='r2')
rf_mse_pca   = neg_mse_to_mse(cross_val_score(rf, X_pca, y_reg, cv=cv_reg, scoring='neg_mean_squared_error'))
rf_r2_pca    = cross_val_score(rf, X_pca, y_reg, cv=cv_reg, scoring='r2')
print(f"RF  NoPCA: MSE={rf_mse_nopca.mean():.4f} R2={rf_r2_nopca.mean():.4f}")
print(f"RF   PCA:  MSE={rf_mse_pca.mean():.4f} R2={rf_r2_pca.mean():.4f}")

In [ ]:
# ── CLASSIFICATION ────────────────────────────────────────────────────────────
# Logistic Regression
log = LogisticRegression(max_iter=2000, C=1.0)
log_acc_nopca = cross_val_score(log, X_scaled, y_cls, cv=cv_cls, scoring='accuracy')
log_f1_nopca  = cross_val_score(log, X_scaled, y_cls, cv=cv_cls, scoring='f1_weighted')
log_acc_pca   = cross_val_score(log, X_pca, y_cls, cv=cv_cls, scoring='accuracy')
log_f1_pca    = cross_val_score(log, X_pca, y_cls, cv=cv_cls, scoring='f1_weighted')
print(f"LogReg NoPCA: Acc={log_acc_nopca.mean():.4f} F1={log_f1_nopca.mean():.4f}")
print(f"LogReg  PCA:  Acc={log_acc_pca.mean():.4f} F1={log_f1_pca.mean():.4f}")

# SVM
svm = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_acc_nopca = cross_val_score(svm, X_scaled, y_cls, cv=cv_cls, scoring='accuracy')
svm_f1_nopca  = cross_val_score(svm, X_scaled, y_cls, cv=cv_cls, scoring='f1_weighted')
svm_acc_pca   = cross_val_score(svm, X_pca, y_cls, cv=cv_cls, scoring='accuracy')
svm_f1_pca    = cross_val_score(svm, X_pca, y_cls, cv=cv_cls, scoring='f1_weighted')
print(f"SVM    NoPCA: Acc={svm_acc_nopca.mean():.4f} F1={svm_f1_nopca.mean():.4f}")
print(f"SVM     PCA:  Acc={svm_acc_pca.mean():.4f} F1={svm_f1_pca.mean():.4f}")

# PCA comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
metrics_names = ['Lin. Reg.\nMSE', 'Lin. Reg.\nR²', 'RF\nMSE', 'RF\nR²']
no_pca_vals = [lr_mse_nopca.mean(), lr_r2_nopca.mean(), rf_mse_nopca.mean(), rf_r2_nopca.mean()]
pca_vals    = [lr_mse_pca.mean(),   lr_r2_pca.mean(),   rf_mse_pca.mean(),   rf_r2_pca.mean()]
x = np.arange(len(metrics_names))
axes[0].bar(x-0.2, no_pca_vals, 0.4, label='No PCA', color='#e74c3c')
axes[0].bar(x+0.2, pca_vals,    0.4, label='PCA',    color='#2980b9')
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics_names)
axes[0].set_title('Regression Metrics comparison'); axes[0].legend()

cls_names = ['LogReg\nAcc', 'LogReg\nF1', 'SVM\nAcc', 'SVM\nF1']
no_pca_cls = [log_acc_nopca.mean(), log_f1_nopca.mean(), svm_acc_nopca.mean(), svm_f1_nopca.mean()]
pca_cls    = [log_acc_pca.mean(),   log_f1_pca.mean(),   svm_acc_pca.mean(),   svm_f1_pca.mean()]
xc = np.arange(len(cls_names))
axes[1].bar(xc-0.2, no_pca_cls, 0.4, label='No PCA', color='#e74c3c')
axes[1].bar(xc+0.2, pca_cls,    0.4, label='PCA',    color='#2980b9')
axes[1].set_xticks(xc); axes[1].set_xticklabels(cls_names)
axes[1].set_title('Classification Metrics comparison'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'pca_comparison.png'))
plt.savefig(os.path.join(PLOT_DIR, 'pca_comparison.eps'))
plt.close()
print("Saved comparison plot.")

In [ ]:
# ── Save results ───────────────────────────────────────────────────────────────
results = {
    'pca_n_components': int(n_components),
    'pca_explained_variance_pct': round(float(explained), 2),
    'regression': {
        'LinearRegression': {
            'mse_nopca': round(float(lr_mse_nopca.mean()), 4),
            'r2_nopca':  round(float(lr_r2_nopca.mean()), 4),
            'mse_pca':   round(float(lr_mse_pca.mean()), 4),
            'r2_pca':    round(float(lr_r2_pca.mean()), 4),
        },
        'RandomForest': {
            'mse_nopca': round(float(rf_mse_nopca.mean()), 4),
            'r2_nopca':  round(float(rf_r2_nopca.mean()), 4),
            'mse_pca':   round(float(rf_mse_pca.mean()), 4),
            'r2_pca':    round(float(rf_r2_pca.mean()), 4),
        }
    },
    'classification': {
        'LogisticRegression': {
            'acc_nopca': round(float(log_acc_nopca.mean()), 4),
            'f1_nopca':  round(float(log_f1_nopca.mean()), 4),
            'acc_pca':   round(float(log_acc_pca.mean()), 4),
            'f1_pca':    round(float(log_f1_pca.mean()), 4),
        },
        'SVM': {
            'acc_nopca': round(float(svm_acc_nopca.mean()), 4),
            'f1_nopca':  round(float(svm_f1_nopca.mean()), 4),
            'acc_pca':   round(float(svm_acc_pca.mean()), 4),
            'f1_pca':    round(float(svm_f1_pca.mean()), 4),
        }
    }
}
with open(r'c:\ML\EXP8\results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Results saved.")